# N-gram Tracing

This notebook will be used to test out n-gram tracing for use with author verification methods. The end goal is to ensure the code works to find common n-grams between two texts and that we can return the text prior to those n-grams.

In [ ]:
import sys

from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from model_loading import load_model
from read_and_write_docs import read_jsonl, read_txt, read_rds
from n_gram_tracing import (
    common_ngrams,
    filter_len_common_ngrams,
    tokens_to_text,
    find_all_ngram_positions,
    texts_before_each_ngram,
    find_all_ngram_spans,
    texts_around_each_ngram
)

In [77]:
tokenizer, model = load_model("/Volumes/BCross/models/gpt2")

In [78]:
known_df = read_jsonl("/Volumes/BCross/datasets/author_verification/test/Wiki/known_raw.jsonl")
unknown_df = read_jsonl("/Volumes/BCross/datasets/author_verification/test/Wiki/unknown_raw.jsonl")
metadata = read_rds('/Volumes/BCross/datasets/author_verification/test/metadata.rds')

In [79]:
problems_str = read_txt("/Volumes/BCross/datasets/author_verification/test/Wiki/agg_problem_list.txt")
problems = problems_str.split("\n")

In [92]:
selected_problem = problems[0]

filtered_metadata = metadata[
    (metadata['corpus'] == 'Wiki')
    & (metadata['problem'] == selected_problem)
].copy()

known_author = filtered_metadata['known_author'].iloc[0]
unknown_author = filtered_metadata['unknown_author'].iloc[0]

selected_known = known_df[known_df['author'] == known_author]
selected_unknown = unknown_df[unknown_df['author'] == unknown_author]

known_text = "\n".join(selected_known["text"])
known_text = selected_known["text"].iloc[2]
unknown_text = selected_unknown['text'].iloc[0]

In [100]:
selected_unknown

,doc_id,text,corpus,author,texttype
0,unknown [Hodja_Nasreddin - Text-3].txt,"If you do not like this article, you are welco...",Wiki,Hodja_Nasreddin,unknown


## Get Common N-Grams

Here we get the n-grams in common between the two texts.

In [93]:
common = common_ngrams(
    text1=known_text,
    text2=unknown_text,
    tokenizer=tokenizer,
    include_subgrams=False,
    lowercase=True
)
common_filtered = filter_len_common_ngrams(common, min_len=None, max_len=None)

In [94]:
common

[['!'],
 ["'"],
 ['as'],
 ['d'],
 ['Ġan'],
 ['Ġare'],
 ['Ġat'],
 ['Ġbecause'],
 ['Ġby'],
 ['Ġcan'],
 ['Ġdone'],
 ['Ġfor'],
 ['Ġhere'],
 ['Ġi'],
 ['Ġin'],
 ['Ġlike'],
 ['Ġmake'],
 ['Ġmuch'],
 ['Ġof'],
 ['Ġone'],
 ['Ġonly'],
 ['Ġprobably'],
 ['Ġshould'],
 ['Ġsite'],
 ['Ġsomething'],
 ['Ġsuch'],
 ['Ġthan'],
 ['Ġthat'],
 ['Ġthe'],
 ['Ġtheir'],
 ['Ġthese'],
 ['Ġthey'],
 ['Ġthink'],
 ['Ġwith'],
 ['Ġwork'],
 ['Ġyou'],
 [',', 'Ġas'],
 ['Ċ', 'if'],
 ['Ċ', 'the'],
 ['Ġdo', 'Ġnot'],
 ['Ġit', '.'],
 ['Ġnew', 'Ġversion'],
 ['Ġthis', 'Ġis'],
 ['Ġto', "Ġ'"],
 ['Ġto', 'Ġbe'],
 ['Ġwe', 'Ġhave'],
 [',', 'Ġbut', 'Ġit'],
 ['.', 'Ċ', 'i'],
 ['.', 'Ċ', 'that'],
 ['Ġwould', 'Ġbe', 'Ġa'],
 [',', 'Ġand', 'Ġso', 'Ġon', '.', 'Ċ']]

In [95]:
common_filtered

[['!'],
 ["'"],
 ['as'],
 ['d'],
 ['Ġan'],
 ['Ġare'],
 ['Ġat'],
 ['Ġbecause'],
 ['Ġby'],
 ['Ġcan'],
 ['Ġdone'],
 ['Ġfor'],
 ['Ġhere'],
 ['Ġi'],
 ['Ġin'],
 ['Ġlike'],
 ['Ġmake'],
 ['Ġmuch'],
 ['Ġof'],
 ['Ġone'],
 ['Ġonly'],
 ['Ġprobably'],
 ['Ġshould'],
 ['Ġsite'],
 ['Ġsomething'],
 ['Ġsuch'],
 ['Ġthan'],
 ['Ġthat'],
 ['Ġthe'],
 ['Ġtheir'],
 ['Ġthese'],
 ['Ġthey'],
 ['Ġthink'],
 ['Ġwith'],
 ['Ġwork'],
 ['Ġyou'],
 [',', 'Ġas'],
 ['Ċ', 'if'],
 ['Ċ', 'the'],
 ['Ġdo', 'Ġnot'],
 ['Ġit', '.'],
 ['Ġnew', 'Ġversion'],
 ['Ġthis', 'Ġis'],
 ['Ġto', "Ġ'"],
 ['Ġto', 'Ġbe'],
 ['Ġwe', 'Ġhave'],
 [',', 'Ġbut', 'Ġit'],
 ['.', 'Ċ', 'i'],
 ['.', 'Ċ', 'that'],
 ['Ġwould', 'Ġbe', 'Ġa'],
 [',', 'Ġand', 'Ġso', 'Ġon', '.', 'Ċ']]

In [37]:
sample_tokens = tokens_to_text(common_filtered[-1], tokenizer)
sample_tokens

', and so on.\n'

## Find Starting Positions

Two options here, to find the starting positions of n-grams and return the text before that or to include the n-gram in the text.

In [38]:
find_all_ngram_positions(unknown_text, sample_tokens)

[927]

In [43]:
example_texts = texts_before_each_ngram(unknown_text, sample_tokens)
for i in range(0, len(example_texts)):
    print(example_texts[i])

If you do not like this article, you are welcome to improve it.
You post your new version of section here, we discuss it, and probably modify it until we come to an agreement that new version is good.
If however you want to create 'new' article
The majority of people think they know something because they read about this in newspapers.
But reality may be very different, as became clear after reading books by experts.
This is especially the case with political controversies in countries with unfree media.
However, this is not the major source of bias in articles.
It is usually assumed that we do not have editorial boards.
It may not be so bad when articles on Physics are owned by physicists except that a person without PhD degree frequently can not understand a thing, but it maybe worse in other areas.
What we have here are fiefdoms when Romanian patriots own Romanian articles, Russian patriots own Russian articles


## Include n-gram in Result

Now we test including the n-gram in the result, either way works since probably apend back on.

In [45]:
common

[['!'],
 ["'"],
 ['as'],
 ['d'],
 ['Ġat'],
 ['Ġby'],
 ['Ġcan'],
 ['Ġcase'],
 ['Ġcountries'],
 ['Ġdid'],
 ['Ġdisputed'],
 ['Ġdone'],
 ['Ġe'],
 ['Ġfor'],
 ['Ġfrom'],
 ['Ġgood'],
 ['Ġhe'],
 ['Ġhere'],
 ['Ġi'],
 ['Ġif'],
 ['Ġlike'],
 ['Ġmake'],
 ['Ġmore'],
 ['Ġmuch'],
 ['Ġonly'],
 ['Ġor'],
 ['Ġother'],
 ['Ġpeople'],
 ['Ġperson'],
 ['Ġprobably'],
 ['Ġshould'],
 ['Ġsimply'],
 ['Ġsite'],
 ['Ġsomething'],
 ['Ġsource'],
 ['Ġsuch'],
 ['Ġthan'],
 ['Ġthat'],
 ['Ġtheir'],
 ['Ġthem'],
 ['Ġthese'],
 ['Ġthey'],
 ['Ġthink'],
 ['Ġwas'],
 ['Ġwhere'],
 ['Ġwill'],
 ['Ġwith'],
 ['Ġwork'],
 ['Ġyour'],
 [',', 'Ġas'],
 [',', 'Ġbecause'],
 [',', 'Ġit'],
 ['Ċ', 'if'],
 ['Ċ', 'the'],
 ['Ġabout', 'Ġthis'],
 ['Ġan', 'Ġagreement'],
 ['Ġarticles', 'Ġon'],
 ['Ġbecause', 'Ġthe'],
 ['Ġexample', ','],
 ['Ġin', 'Ġa'],
 ['Ġit', '.'],
 ['Ġnew', 'Ġversion'],
 ['Ġr', 'ussian'],
 ['Ġthe', 'Ġsubject'],
 ['Ġthis', 'Ġarticle'],
 ['Ġto', "Ġ'"],
 ['Ġto', 'Ġbe'],
 ['Ġto', 'Ġthe'],
 ['Ġvery', 'Ġdifferent'],
 ['Ġwe', 'Ġhave'],
 [',', 

In [47]:
find_all_ngram_spans(known_text, "as")

[(258, 260),
 (406, 408),
 (409, 411),
 (650, 652),
 (666, 668),
 (1131, 1133),
 (1135, 1137),
 (1198, 1200),
 (1276, 1278),
 (1666, 1668),
 (1718, 1720),
 (1798, 1800),
 (1941, 1943),
 (2299, 2301),
 (2303, 2305),
 (2852, 2854),
 (2857, 2859),
 (3005, 3007),
 (3059, 3061),
 (3089, 3091),
 (3117, 3119),
 (3173, 3175),
 (3307, 3309),
 (3326, 3328),
 (3402, 3404),
 (3696, 3698),
 (3879, 3881),
 (4268, 4270),
 (4492, 4494),
 (4603, 4605),
 (4670, 4672),
 (4678, 4680),
 (4703, 4705)]

In [55]:
example_texts = texts_around_each_ngram(known_text, " as")

for i in range(0, len(example_texts)):
    print(f"Text {i} - {example_texts[i][-20:]}")

Text 0 - tions about Kirov as
Text 1 - what sources tell as
Text 2 - les accordingly - as
Text 3 -  these countries, as
Text 4 - m.
Taking lipases as
Text 5 -  should be split, as
Text 6 - ses that were not as
Text 7 - queries is a huge as
Text 8 - heoretical models as
Text 9 - o keep everything as
Text 10 - is should be done as
Text 11 - ng independently, as
Text 12 - endently, as soon as


In [101]:
example_texts = texts_around_each_ngram(known_text, ['as'])

for i in range(0, len(example_texts)):
    print(f"Text {i} - {example_texts[i][-20:]}")

AttributeError: 'list' object has no attribute 'casefold'

In [97]:
example_texts = texts_around_each_ngram(unknown_text, 'as')

for i in range(0, len(example_texts)):
    print(f"Text {i} - {example_texts[i][-20:]}")

Text 0 - e very different, as
Text 1 - s especially the cas
Text 2 - major source of bias
Text 3 - es.
It is usually as
Text 4 - worse in other areas
Text 5 - ea or two teams clas
Text 6 - major source of bias
Text 7 - ooks that qualify as
Text 8 - ndidates to vote.
As
Text 9 - te links to, just as
Text 10 - g to be me.
That was
